In [ ]:
#Para el entrenamiento
#pip install wandb

In [ ]:
import re
import os
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import Dataset, DatasetDict
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt


column_names = ['label', 'statement', 'subject', 'speaker', 'job_title', 'state', 'party',
                'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts',
                'pants_on_fire_counts', 'context']

train_df = pd.read_csv("train.tsv", sep='\t', names=column_names).dropna(subset=["label", "statement"])
valid_df = pd.read_csv("valid.tsv", sep='\t', names=column_names).dropna(subset=["label", "statement"])
test_df = pd.read_csv("test.tsv", sep='\t', names=column_names).dropna(subset=["label", "statement"])

# Mapeo de etiquetas a números
label_map = {
    'pants-fire': 0,
    'false': 1,
    'barely-true': 2,
    'half-true': 3,
    'mostly-true': 4,
    'true': 5
}

train_df['label'] = train_df['label'].map(label_map)
valid_df['label'] = valid_df['label'].map(label_map)
test_df['label'] = test_df['label'].map(label_map)


#Preprocesamiento del dataset
#Limpieza básica del texto
def limpiar_texto(texto):
    texto = texto.lower()  # minúsculas
    texto = re.sub(r"http\S+", "", texto)  # quitar URLs
    texto = re.sub(r"[^a-zA-Z0-9\s]", "", texto)  # quitar símbolos raros
    texto = re.sub(r"\s+", " ", texto).strip()  # quitar espacios extras
    return texto

# Aplicar al dataframe
for df in [train_df, valid_df, test_df]:
    df["statement"] = df["statement"].apply(limpiar_texto)

train_df["statement"].sample(5)

In [ ]:
#Convertir a formato Dataset de HuggingFace
train_ds = Dataset.from_pandas(train_df[['statement', 'label']], preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df[['statement', 'label']], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['statement', 'label']], preserve_index=False)

dataset = DatasetDict({
    'train': train_ds,
    'validation': valid_ds,
    'test': test_ds
})

#Tokenización
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(example):
    return tokenizer(example["statement"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

#Definir el modelo
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6)

#Configurar Trainer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1": f1_score(p.label_ids, preds, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",                # <--- aquí el cambio importante
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
)
#Desactiva WandB para que no pida login
os.environ["WANDB_DISABLED"] = "true"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

#Entrenamiento del modelo
trainer.train()

#Evaluación en test
metrics = trainer.evaluate(tokenized_datasets["test"])
print(metrics)

In [ ]:
predictions_output = trainer.predict(tokenized_datasets["test"])

# Predicciones como etiquetas (0–5)
predicted_labels = np.argmax(predictions_output.predictions, axis=1)

# Reales
true_labels = predictions_output.label_ids

# Mapear valores numéricos a nombres de clase
label_map_inv = {
    0: 'pants-fire',
    1: 'false',
    2: 'barely-true',
    3: 'half-true',
    4: 'mostly-true',
    5: 'true'
}

# Aplicar el mapeo a la columna de etiquetas reales
etiquetas = test_df['label'].map(label_map_inv)

# Gráfico de barras
plt.figure(figsize=(8,5))
sns.countplot(x=etiquetas, order=label_map_inv.values())
plt.title('Distribución real de etiquetas en el test set (multiclase)')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)
plt.show()


#Matriz de confusión
cm = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de confusión - Multiclase')
plt.show()


# Reporte completo
print(classification_report(true_labels, predicted_labels, target_names=[
    "pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"
]))